In [ ]:
# ==============================================================
# CELL 1 – Setup
# ==============================================================
import os
import sys
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torch.utils.data import TensorDataset, DataLoader, Dataset
from torch.optim import AdamW
from tqdm import tqdm

# --- PARAMETRI AMBIENTE HPC ---
DATA_PATH = os.environ.get('DATA_PATH', 'data/PreTrain/SUPER_PRETRAIN_500pt.npz')
OUTPUT_DIR = os.environ.get('OUTPUT_DIR', 'experiments/smae_pretrain/500_pt/exp_1/')
EPOCHS = int(os.environ.get('EPOCHS', '50'))
os.makedirs(OUTPUT_DIR, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'PyTorch: {torch.__version__}')
torch.manual_seed(42)
np.random.seed(42)
print(f'Dati da caricare: {DATA_PATH}')
print(f'Output dir: {OUTPUT_DIR}')
print(f'Epoche: {EPOCHS}')


In [2]:
# ==========================================
# CELL 2: DATASET DEFINITION
# ==========================================
class RamanPretrainDataset(Dataset):
    def __init__(self, data_path):
        super().__init__()
        print(f"Opening file: {data_path}")
        if data_path.endswith('.mat'):
            import h5py
            with h5py.File(data_path, 'r') as f:
                self.X = np.array(f['X_SUPER_PRETRAIN']).T.astype(np.float32)
                if 'dataset_source' in f:
                    self.sources = np.array(f['dataset_source']).flatten().astype(np.int64)
                else:
                    self.sources = np.zeros(len(self.X), dtype=np.int64)
        else:
            loaded_data = np.load(data_path)
            self.X = loaded_data['X_SUPER_PRETRAIN'].astype(np.float32)
            if 'dataset_source' in loaded_data:
                self.sources = loaded_data['dataset_source'].flatten().astype(np.int64)
            else:
                self.sources = np.zeros(len(self.X), dtype=np.int64)
        print(f"Data loaded. Total spectra: {self.X.shape[0]}")

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return torch.from_numpy(self.X[idx]).unsqueeze(0), self.sources[idx]

dataset = RamanPretrainDataset(DATA_PATH)
train_loader = DataLoader(dataset, batch_size=256, shuffle=True, num_workers=2, pin_memory=True)


In [ ]:
# ==============================================================
# CELL 3 - Import SpectraMAENet & Build Model
# ==============================================================
import os
import sys
import torch

# Usa il file locale invece di scaricarlo da GitHub
sys.path.append('./src/models/transformer')
from Spectra_MAE import MaskedAutoencoderViT

# -----------------------------------------------------------
# 3. Hyperparameters
# -----------------------------------------------------------
SPECTRA_SIZE   = 1000
PATCH_SIZE     = 50
ENCODER_DIM    = 256
ENCODER_DEPTH  = 8
ENCODER_HEADS  = 8
DIM_MLP        = 512
DECODER_DIM    = 128
DECODER_DEPTH  = 2
DECODER_HEADS  = 8
MASK_RATIO     = 0.75

def build_smae():
    return MaskedAutoencoderViT(
        spectra_size       = SPECTRA_SIZE,
        patch_size         = PATCH_SIZE,
        encoder_dim        = ENCODER_DIM,
        depth              = ENCODER_DEPTH,
        num_heads          = ENCODER_HEADS,
        dim_mlp            = DIM_MLP,
        decoder_embed_dim  = DECODER_DIM,
        decoder_depth      = DECODER_DEPTH,
        decoder_num_heads  = DECODER_HEADS,
        norm_pix_loss      = False,
    ).to(device)

print("Building the SMAE model...")
model = build_smae()

x_test = torch.randn(4, 1, SPECTRA_SIZE).to(device)
loss_test, _, _ = model(x_test, mask_ratio=MASK_RATIO)

n_params  = sum(p.numel() for p in model.parameters())
n_patches = SPECTRA_SIZE // PATCH_SIZE

print("\n--------------------------------------------------")
print(f'Total Parameters: {n_params:,}')
print(f'Total Patches:    {n_patches}  ({SPECTRA_SIZE} / {PATCH_SIZE})')
print(f'Test Loss:        {loss_test.item():.5f}')
print("--------------------------------------------------")


In [4]:
# ==============================================================
# CELL 4: RECONSTRUCTION VISUALIZER & SAVER
# ==============================================================
import os
import matplotlib.pyplot as plt
import torch

# Mappa le etichette ai nomi dei dataset originari
SOURCE_MAP = {
    1: 'TropHY',
    2: 'DeepR (Noisy)',
    3: 'CELLS',
    4: 'TWIST',
    5: 'IMMUNO'
}

def visualize_progress(model, fixed_batch, fixed_sources, device, epoch):
    import os
    import matplotlib.pyplot as plt
    import numpy as np
    import torch

    PLOT_DIR = OUTPUT_DIR
    os.makedirs(PLOT_DIR, exist_ok=True)

    model.eval()
    batch = fixed_batch.to(device)

    with torch.no_grad():
        loss, pred, mask = model(batch, mask_ratio=MASK_RATIO)

    # Shapes:
    # batch -> (B, 1, 500)
    # pred  -> (B, num_patches, patch_size)
    # mask  -> (B, num_patches)

    orig = batch.cpu().squeeze(1).numpy()   # (B, 500)
    pred = pred.cpu().numpy()               # (B, 20, 25)
    mask = mask.cpu().numpy()               # (B, 20)
    sources = fixed_sources.cpu().numpy()

    num_samples = len(orig)
    plt.figure(figsize=(14, 3.5 * num_samples))

    for i in range(num_samples):
        x_orig = orig[i].copy()
        x_recon = x_orig.copy()

        ax = plt.subplot(num_samples, 1, i + 1)

        # Rebuild the final spectrum:
        # replace ONLY masked patches with decoder predictions
        for p, is_masked in enumerate(mask[i]):
            start = p * PATCH_SIZE
            end = start + PATCH_SIZE

            if is_masked:
                x_recon[start:end] = pred[i, p]

                # Highlight masked regions on the plot
                ax.axvspan(start, end, color='khaki', alpha=0.35)

        # Plot original and reconstruction
        ax.plot(x_orig, label='Original Spectrum', color='royalblue', alpha=0.8)
        ax.plot(x_recon, label='MAE Reconstruction', color='red', linestyle='--', linewidth=1.5)

        # Recupera il nome del dataset originario
        source_name = SOURCE_MAP.get(int(sources[i]), "Unknown")

        if i == 0:
            ax.set_title(f'Reconstruction Progress – Epoch {epoch} | Source: {source_name}', fontsize=14)
        else:
            ax.set_title(f'Source: {source_name}', fontsize=14)

        ax.set_xlabel('Spectral Point Index')
        ax.set_ylabel('Intensity (L2-normalized)')
        ax.grid(True, alpha=0.3)
        ax.legend(loc='upper right')

    plt.tight_layout()

    save_path = os.path.join(PLOT_DIR, f'Reconstruction_Epoch_{epoch:03d}.png')
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()

    print(f'📸 Plot saved successfully to: {save_path}')

    model.train()

In [5]:
# ==============================================================
# HELPER FUNCTION – Masked Cosine Similarity
#
# OBJECTIVE:
#   Compute cosine similarity ONLY on the MASKED regions,
#   because the MAE is trained to reconstruct the hidden patches.
#
# WHY:
#   Using cosine on the full spectrum can be misleading, because
#   it also includes visible regions that the model did not need
#   to reconstruct.
# ==============================================================

import torch

def masked_cosine_similarity(target, pred, mask, patch_size, eps=1e-8):
    """
    Computes cosine similarity ONLY on masked regions.

    Args:
        target: Tensor of shape (B, 1, 500) or (B, 500)
        pred:   Tensor of shape (B, num_patches, patch_size)
        mask:   Tensor of shape (B, num_patches), where 1 = masked patch
        patch_size: number of spectral points per patch
        eps: numerical stability constant

    Returns:
        Mean cosine similarity over the batch
    """

    # Remove channel dimension if needed: (B, 1, 500) -> (B, 500)
    if target.dim() == 3:
        target = target.squeeze(1)

    # Flatten decoder predictions: (B, num_patches, patch_size) -> (B, 500)
    pred_flat = pred.flatten(1)

    # Expand patch mask to point-level mask: (B, 20) -> (B, 500)
    mask_expanded = mask.unsqueeze(-1).repeat(1, 1, patch_size).flatten(1).float()

    # Keep only masked regions
    target_masked = target * mask_expanded
    pred_masked   = pred_flat * mask_expanded

    # Manual cosine similarity for numerical stability
    dot_product = (target_masked * pred_masked).sum(dim=1)
    target_norm = target_masked.norm(dim=1)
    pred_norm   = pred_masked.norm(dim=1)

    cos_sim = dot_product / (target_norm * pred_norm + eps)

    return cos_sim.mean()

In [ ]:
# ==============================================================
# CELL 5 – Pre-Training SMAE Loop
# ==============================================================

import os
import torch
import torch.nn.functional as F
import numpy as np

# Path to save the final Foundation Model weights
PATH_PRETRAINED = os.path.join(OUTPUT_DIR, 'smae_pretrained_model.pth')
os.makedirs(os.path.dirname(PATH_PRETRAINED), exist_ok=True)

# Learning rate for pre-training
PRETRAIN_LR     = 1e-4

# --------------------------------------------------------------
# PREPARATION OF A FIXED BATCH FOR VISUALIZATION
# We select 1 spectrum from TropHY (1), 1 from DeepR (2) and 1 from TWIST (4)
# --------------------------------------------------------------
print("Selecting fixed spectra for visualization...")
fixed_indices = []
desired_sources = [1, 2, 4] # Ensure we include DeepR (2)

for target_src in desired_sources:
    # Find the first index in the dataset for this source
    idx = np.where(dataset.sources == target_src)[0]
    if len(idx) > 0:
        fixed_indices.append(idx[0])

# Build the fixed tensors for visualization
fixed_batch_list = []
fixed_sources_list = []
for idx in fixed_indices:
    x, y = dataset[idx]
    fixed_batch_list.append(x)
    fixed_sources_list.append(torch.tensor(y))

fixed_batch = torch.stack(fixed_batch_list)
fixed_sources = torch.stack(fixed_sources_list)
print(f"Fixed spectra ready! Sources: {[int(s) for s in fixed_sources]}")


print(f'\nStarting Pre-training for {EPOCHS} epochs...\n')

optimizer_pre = torch.optim.AdamW(model.parameters(), lr=PRETRAIN_LR, weight_decay=1e-4)
scheduler_pre = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_pre, T_max=EPOCHS)

# Tracking both metrics
pretrain_losses = []
pretrain_cos_sims = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    total_cos_sim = 0.0

    # NOTE: The DataLoader now returns (spectrum, source)
    # We loop directly over train_loader to avoid tqdm spam on HPC logs
    for batch_data, batch_sources in train_loader:
        batch_data = batch_data.to(device)

        optimizer_pre.zero_grad()

        # Forward pass
        loss, pred, mask = model(batch_data, mask_ratio=MASK_RATIO)

        # ----------------------------------------------------------
        # CORRECT COSINE SIMILARITY:
        # ----------------------------------------------------------
        cos_sim = masked_cosine_similarity(
            target=batch_data,
            pred=pred,
            mask=mask,
            patch_size=PATCH_SIZE
        )

        # Backward pass and optimization
        loss.backward()
        optimizer_pre.step()

        total_loss += loss.item()
        total_cos_sim += cos_sim.item()

    # Calculate averages for the entire epoch
    avg_loss = total_loss / len(train_loader)
    avg_cos_sim = total_cos_sim / len(train_loader)

    pretrain_losses.append(avg_loss)
    pretrain_cos_sims.append(avg_cos_sim)
    scheduler_pre.step()

    # Print metrics to screen at EVERY epoch to monitor progress (replaces tqdm)
    print(f'Epoch {epoch:3d}/{EPOCHS} | MSE Loss: {avg_loss:.6f} | Masked Cosine Sim: {avg_cos_sim:.4f}')
    
    # Visualize plots using our FIXED SPECTRA every 10 epochs (and the 1st epoch)
    if epoch == 1 or epoch % 10 == 0:
        visualize_progress(model, fixed_batch, fixed_sources, device, epoch)

# Final save of the Foundation Model weights
torch.save(model.state_dict(), PATH_PRETRAINED)
print(f'\n✅ Pre-trained foundation weights successfully saved -> {PATH_PRETRAINED}')

In [ ]:
# ==============================================================
# CELL 6 – LEARNING CURVES PLOT & FINAL CHECKS
# ==============================================================
import matplotlib.pyplot as plt
import os

PLOT_DIR = os.path.join(OUTPUT_DIR, 'Report_Plots')
os.makedirs(PLOT_DIR, exist_ok=True)

# Create a figure with 2 subplots (1 row, 2 columns)
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('SMAE Pre-Training Performance Metrics', fontsize=16, fontweight='bold', y=1.05)

# --- Subplot 1: MSE Loss (Should go DOWN) ---
axes[0].plot(range(1, len(pretrain_losses) + 1), pretrain_losses, marker='o', color='red', markersize=4)
axes[0].set_title('Reconstruction Loss', fontsize=14)
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Mean Squared Error (MSE)', fontsize=12)
axes[0].grid(True, linestyle='--', alpha=0.7)

# --- Subplot 2: Cosine Similarity (Should go UP towards 1.0) ---
axes[1].plot(range(1, len(pretrain_cos_sims) + 1), pretrain_cos_sims, marker='s', color='green', markersize=4)
axes[1].set_title('Shape Alignment (Cosine Similarity)', fontsize=14)
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Cosine Similarity (1.0 = Perfect)', fontsize=12)
axes[1].grid(True, linestyle='--', alpha=0.7)

plt.tight_layout()

# Save the double plot in high resolution
curve_path = os.path.join(PLOT_DIR, 'PreTraining_Metrics_DoublePlot.png')
plt.savefig(curve_path, dpi=300, bbox_inches='tight')

# Display the plot
plt.show()

print(f"📈 Performance Metrics plot saved to: {curve_path}")
print("==============================================================")
print(" 🚀 PRE-TRAINING PHASE COMPLETELY FINISHED! READY FOR FINE-TUNING.")
print("==============================================================")

In [ ]:
# ==============================================================
# CELL 7 – POST-TRAINING EVALUATION (TropHY, DeepR, IMMUNO)
# ==============================================================
import torch
import numpy as np
import random

print("Selezione casuale di un nuovo set di spettri per la valutazione finale...")
# 1: TropHY, 2: DeepR, 4: TWIST, 5: IMMUNO
desired_sources_post = [1, 2, 4]
fixed_indices_post = []

for target_src in desired_sources_post:
    # Trova gli indici nel dataset con questa provenienza
    idx = np.where(dataset.sources == target_src)[0]
    if len(idx) > 0:
        # Sceglie un indice casuale, così puoi rieseguire la cella finché non trovi
        # gli spettri ideali (es. un DeepR più rumoroso)
        random_idx = np.random.choice(idx)
        fixed_indices_post.append(random_idx)

# Costruiamo i tensori
fixed_batch_post_list = []
fixed_sources_post_list = []

for idx in fixed_indices_post:
    x, y = dataset[idx]
    fixed_batch_post_list.append(x)
    fixed_sources_post_list.append(torch.tensor(y))

fixed_batch_post = torch.stack(fixed_batch_post_list)
fixed_sources_post = torch.stack(fixed_sources_post_list)

print(f"Spettri finali pronti! Provenienza: {[int(s) for s in fixed_sources_post]}")
print(f"Indici selezionati casualmente: {fixed_indices_post}")

# Assicuriamoci che il modello sia in modalità valutazione
model.eval()

# Chiamiamo la funzione di visualizzazione (passiamo PRETRAIN_EPOCHS come 'epoca' per il titolo e nome file)
visualize_progress(model, fixed_batch_post, fixed_sources_post, device, EPOCHS)